In [1]:
import pandas as pd 
import duckdb
import os
import requests
from datetime import datetime
from dotenv import load_dotenv , find_dotenv
import math 

# 1. 환경 변수 로드
load_dotenv(find_dotenv())
MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')

# 1. 환경 변수 로드
load_dotenv(find_dotenv())
OPINET_API_KEY_GROUP = ['OPINET_API_KEY_1', 'OPINET_API_KEY_2', 'OPINET_API_KEY_3', 
                        'OPINET_API_KEY_4', 'OPINET_API_KEY_5', 'OPINET_API_KEY_6' ] 
MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT') 
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')


#2. duckdb를 통한 s3 읽기 설정
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"SET s3_endpoint='{MINIO_ENDPOINT}';")
con.execute(f"SET s3_access_key_id='{MINIO_ACCESS_KEY}';")
con.execute(f"SET s3_secret_access_key='{MINIO_SECRET_KEY}';")
con.execute("SET s3_url_style='path'; SET s3_use_ssl='false';")


print("✅ DuckDB의 MinIO 접속 준비 완료!")



✅ DuckDB의 MinIO 접속 준비 완료!


In [2]:

df = con.sql(
    """
    select 
    uni_cd
    , gasoline
    , diesel
    , premium_gasoline
    from read_parquet('s3://petroleum-project/station_price/agg/*/*.parquet')
    where gasoline is not null and diesel is not null and premium_gasoline is not null  -- 첫번째 샘플은 걍 모든 데이터 있는 것으로.
    and part_dt = '20260425'
    order by uni_cd
    limit 2

"""
).df()

display(df)

,uni_cd,gasoline,diesel,premium_gasoline
0,A0000004,2009,2001,2388
1,A0000011,1984,1984,2385


In [3]:
# 체득 1 모든 과정을 직접 계산하기

i = 0
A = df.iloc[0][['gasoline' , 'diesel' , 'premium_gasoline']]
B = df.iloc[1][['gasoline' , 'diesel' , 'premium_gasoline']]

dot = (A*B).sum()
display(dot)

13651220

In [4]:
# dot product는 수기로 계산했으니, 각각의 거리를 계산하자.
len_a = math.sqrt((df.iloc[0][['gasoline' , 'diesel' , 'premium_gasoline']]**2).sum())
len_b = math.sqrt((df.iloc[1][['gasoline' , 'diesel' , 'premium_gasoline']]**2).sum())

# 둘다 잘 되지만 math.sqrt는 series에 대한 계산이 안된다는 치명적인 단점이 있다 그러므로 **0.5 가 낫겠어. ** (1/2) 이 조금더 뭐랄까... 난 좋게 느껴져. 수학의 정석책을 보는 느낌이야.

norm_a = (df.iloc[0][['gasoline' , 'diesel' , 'premium_gasoline']]**2).sum()**(1/2) #용어도 많이 쓰는 norm으로 바꾸자
norm_b = (df.iloc[1][['gasoline' , 'diesel' , 'premium_gasoline']]**2).sum()**(1/2)
display(norm_a)
display(norm_b)

3707.104800245065

3682.490597408227

In [5]:
# cosine similarity 의 공식 =  A dot B / |A| * |B| = cosine theta

cos_theta = dot / (norm_a * norm_b)
display(cos_theta)


0.9999883842859872

**이 Cosine Similarity를 공부하는 5가지 방법**

1. sklearn `cosine_similarity` 함수 사용 --> Output : N*N Matrix
2. 1에서 만든 N*N Matrix를 Long format으로 만들기 (Pandas 활용)
3. Top-K에 대해서 Long format으로 `cosine_similarity` 를 계산해주는 함수 사용해보기
4. Duck DB의 내장된 `cosine_similarity` 함수를 활용해보기
5. Cross-join으로 모든 SQL에서 작동하는 방식으로 계산해보기



In [6]:

df = con.sql(
    """
    select 
    uni_cd
    , gasoline
    , diesel
    , premium_gasoline
    from read_parquet('s3://petroleum-project/station_price/agg/*/*.parquet')
    where gasoline is not null -- and diesel is not null and premium_gasoline is not null  -- 첫번째 샘플은 걍 모든 데이터 있는 것으로.
    and part_dt = '20260425'
    order by uni_cd 
"""
).df()

display(df)

,uni_cd,gasoline,diesel,premium_gasoline
0,A0000004,2009,2001,2388
1,A0000011,1984,1984,2385
2,A0000012,2011,1979,2383
3,A0000014,2026,2020,2412
4,A0000015,2022,2009,<NA>
...,...,...,...,...
10293,A0033812,1990,1970,<NA>
10294,A0033813,1988,1984,2319
10295,A0033814,1988,1986,2319
10296,A0033816,1980,2000,<NA>


In [7]:
from sklearn import metrics

cos_df = df.set_index('uni_cd' )    #df[['gasoline' , 'diesel' , 'premium_gasoline']]
cos_df = cos_df.fillna(0)
cos_similarity = metrics.pairwise.cosine_similarity(cos_df)

#display(cos_similarity)
#display(cos_df.median())
#display(cos_df.mean())

sim_wide = pd.DataFrame(
    cos_similarity,
    index = cos_df.index ,
    columns = cos_df.index
)
#display(sim_wide)
sim_long = (
            sim_wide
            .stack()
            .rename_axis(['uni_cd_a','uni_cd_b' ])
            .reset_index(name = 'cos_similarity')
)
sim_clean = (
    sim_long[sim_long['uni_cd_a'] < sim_long['uni_cd_b']]
    .sort_values(by='cos_similarity', ascending = False)
    .head(20)
)
display(sim_clean)

,uni_cd_a,uni_cd_b,cos_similarity
24091666,A0008736,A0016318,1.0
36833648,A0012699,A0027158,1.0
36833668,A0012699,A0027235,1.0
36833666,A0012699,A0027226,1.0
10150889,A0003804,A0025206,1.0
65587216,A0021855,A0032045,1.0
14370058,A0005329,A0015337,1.0
46316543,A0015779,A0022055,1.0
46316542,A0015779,A0022052,1.0
10150896,A0003804,A0025218,1.0


In [15]:
from sklearn.neighbors import NearestNeighbors

cos_df = df.set_index('uni_cd' )
cos_df = cos_df.fillna(0)
nn = NearestNeighbors ( n_neighbors = 20 , algorithm = 'auto',metric='cosine' )
nn.fit(cos_df)


n_samples = nn.n_samples_fit_
display(n_samples)

10298

In [20]:
distances, indices = nn.kneighbors(cos_df)

n,k = distances.shape 

cos_long = pd.DataFrame({
    'uni_cd_a' : cos_df.index.repeat(k),
    'uni_cd_b' : cos_df.index.values[indices.flatten()],
    'cos_similarity' : 1 - distances.flatten(),
})

cos_long = cos_long[cos_long['uni_cd_a'] != cos_long['uni_cd_b']]
display(cos_long)

,uni_cd_a,uni_cd_b,cos_similarity
1,A0000004,A0016227,1.0
2,A0000004,A0014138,1.0
3,A0000004,A0010053,1.0
4,A0000004,A0002002,1.0
5,A0000004,A0031986,1.0
...,...,...,...
205955,A0033818,A0005091,1.0
205956,A0033818,A0008740,1.0
205957,A0033818,A0001022,1.0
205958,A0033818,A0027531,1.0
